In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
from dotenv import load_dotenv

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
client = QdrantClient("localhost", port=6333)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9174.28it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype="auto", trust_remote_code=True
)

# Create a generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 340/340 [00:03<00:00, 101.97it/s]


In [7]:
query = "Yayaların hakları nelerdir?"

In [8]:
# 1. Encode with E5-Small (adding the required prefix)
query_text = f"query: {query}"
query_emb = embed_model.encode([query_text])[0]

In [9]:
# 2. Use 'query_points' instead of 'search'
results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=5
).points  # Note: .points is needed to get the list of results

In [10]:
# 3. Extract text from the payload
context = "\n\n".join(r.payload["text"] for r in results)

In [13]:
# 4. Generate with Gemma
messages = [
    {"role": "system", "content": "Sen bir Türk hukuku asistanısın."},
    {"role": "user", "content": f"Bağlam:\n{context}\n\nSoru: {query}"},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
outputs = pipe(
    prompt,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7,
    return_full_text=False,
)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [14]:
print(outputs[0]["generated_text"].strip())

Yayaların hakları şunlardır:

*   **Taşıt Yolu Üzerinde Yürümek:** Anayasa Mahkemesi'nin 14/3/2019 tarihli E. 2019/1, K. 2019/14 sayılı kararı ile bu hak, 5.010'a göre "5.010" ibaresi "140.000" şeklinde değiştirilmiştir.
*   **Yaya Yolu ve Banketlerin Kullanılması:** Yayalar, taşıt yolu bitişiğinde ve yakınında yaya yolu, banket bulunmayan veya kullanılır durumda olan karayollarda, yarmaca, şevden sonra hendek varsa hendek dış kenarı, hendek yoksa şev üst kenarı, dolguda şev etek çizgisi, yaya ayrılmış karayolu üzerinde yaya yolunun mülkle birleştiği çizgidir.
*   **Taşit Yolu Üzerinde Yürüme:** Yayalar, taşıt yolunun her iki yönündeki trafiğin kullanıldığı karayollarında, yaya ve okul geçidi ile bir yetkili veya görevli yönetiminde yürüyebilirler.
*   **Yaya Geçitlerinde ve Kavşaklarda Yürüme:** Yaya geçitlerinde ve kavşaklarda yürüme zorundadırlar.
*   **Yaya Kafileleri Yürümek:** Bir yetkili veya görevli yönetiminde yürüyen yaya kafileleri taşıt yolu üzerinde yürüyebilirler.
*   **T